In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import os
import sys
import time 

from pathlib import Path
from loguru import logger

import duckdb

In [25]:
ROOT = Path.cwd().parent
RAW_DATA_DIR = ROOT / "data" / "raw"
OUTPUT_DATA_DIR = ROOT / "data" / "interim"
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
con = duckdb.connect()
csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))
for csv_file in csv_files:
    parquet_file = OUTPUT_DATA_DIR / f"{csv_file.stem}.parquet"

    # Skip conversion if the Parquet file already exists
    if parquet_file.exists():
        logger.info(f"Skipping {csv_file} as {parquet_file} already exists.")
        continue

    start_time = pd.Timestamp.now()
    con.execute(f"COPY (SELECT * FROM read_csv_auto('{csv_file}')) TO '{parquet_file}' (FORMAT PARQUET);")
    end_time = pd.Timestamp.now()

    logger.info(f"Converted {csv_file} to {parquet_file} in {end_time - start_time}")
    old_size = csv_file.stat().st_size/ (1024 * 1024)
    new_size = parquet_file.stat().st_size/ (1024 * 1024)
    logger.info(f"Size reduced from {old_size:.2f} MB to {new_size:.2f} MB ({(old_size - new_size) / old_size * 100:.2f}% reduction)")

con.close()


2026-10-10 21:36:00.566 | INFO     | __main__:<module>:10 - Converted /Users/yuzhezhang/Library/Mobile Documents/com~apple~CloudDocs/MyProject/data_science/data/raw/clients.csv to /Users/yuzhezhang/Library/Mobile Documents/com~apple~CloudDocs/MyProject/data_science/data/interim/clients.parquet in 0 days 00:00:00.130420
2026-10-10 21:36:00.567 | INFO     | __main__:<module>:13 - Size reduced from 20.73 MB to 9.51 MB (54.11% reduction)
2026-10-10 21:36:00.657 | INFO     | __main__:<module>:10 - Converted /Users/yuzhezhang/Library/Mobile Documents/com~apple~CloudDocs/MyProject/data_science/data/raw/products.csv to /Users/yuzhezhang/Library/Mobile Documents/com~apple~CloudDocs/MyProject/data_science/data/interim/products.parquet in 0 days 00:00:00.089297
2026-10-10 21:36:00.657 | INFO     | __main__:<module>:13 - Size reduced from 3.71 MB to 0.87 MB (76.53% reduction)
2026-10-10 21:36:06.960 | INFO     | __main__:<module>:10 - Converted /Users/yuzhezhang/Library/Mobile Documents/com~apple~

# Checking the Quality of the Data

After the conversion is finished, the next step is to check the quality of the data. In the real world, data is often dirty: some values are missing, some are incorrect, and some columns have the wrong data types.

In [32]:
con = duckdb.connect()

parquet_files = sorted(OUTPUT_DATA_DIR.glob("*.parquet"))

summary = []
for parquet_file in parquet_files:
    relation = con.read_parquet(str(parquet_file))

    summary.append({
        "table": parquet_file.stem,
        "rows": relation.shape[0],
        "columns": relation.shape[1],
    })

summary_df = pd.DataFrame(summary)
summary_df

,table,rows,columns
0,clients,400162,5
1,products,43038,11
2,purchases,45786568,13
3,uplift_sample_submission,200123,2
4,uplift_test,200123,1
5,uplift_train,200039,3


In [34]:
for parquet_file in parquet_files:
    relation = con.read_parquet(str(parquet_file))

    print("=" * 30)
    print(f"Table: {parquet_file.stem}")
    schema_df = pd.DataFrame({
        "column_name": relation.columns,
        "data_type": [str(dtype) for dtype in relation.dtypes]
    })

    display(schema_df)

Table: clients


,column_name,data_type
0,client_id,VARCHAR
1,first_issue_date,TIMESTAMP
2,first_redeem_date,TIMESTAMP
3,age,BIGINT
4,gender,VARCHAR


Table: products


,column_name,data_type
0,product_id,VARCHAR
1,level_1,VARCHAR
2,level_2,VARCHAR
3,level_3,VARCHAR
4,level_4,VARCHAR
5,segment_id,DOUBLE
6,brand_id,VARCHAR
7,vendor_id,VARCHAR
8,netto,DOUBLE
9,is_own_trademark,BIGINT


Table: purchases


...

,column_name,data_type
0,client_id,VARCHAR
1,uplift,DOUBLE


Table: uplift_test


,column_name,data_type
0,client_id,VARCHAR


Table: uplift_train


,column_name,data_type
0,client_id,VARCHAR
1,treatment_flg,BIGINT
2,target,BIGINT
